# XSL Figure 1 reference-star validation

This notebook tests Spyctres against the nine OBAFGKMC reference spectra shown in Figure 1 of [Verro et al. (2022)](https://doi.org/10.1051/0004-6361/202142388). It compares fitted $T_{\rm eff}$, $\log g$, and $[\mathrm{Fe/H}]$ with the XSL values from [Arentsen et al. (2019)](https://doi.org/10.1051/0004-6361/201834273), Verro et al. (2022), and the carbon-star analysis of [Gonneau et al. (2017)](https://doi.org/10.1051/0004-6361/201629750).

XSL DR3 spectra were observed with X-SHOOTER, but the released DR3 files are not the same product as a generic reduced X-SHOOTER arm spectrum. They are combined-arm, logarithmically sampled, air-wavelength, stellar-rest-frame library products with documented arm/overlap resolution metadata and header provenance for rest-frame/arm scaling. Spyctres therefore reads them with the explicit `instrument="xsl_dr3"` identifier, while still converting them to the same canonical `SpectrumCollection` structure used by the X-SHOOTER examples.

This is a scientific validation, not a nine-star pass/fail contest. [Lançon et al. (2021)](https://doi.org/10.1051/0004-6361/202039371) found arm-dependent PHOENIX/XSL offsets and increasingly structured residuals below about 5000 K. Three objects deliberately probe boundaries: X0116 is hotter than the supported 12000 K PHOENIX grid and is not fitted; X0013 is carbon-rich but the current fit has no C/O parameter; X0119 is a very cool, low-gravity supergiant for which hydrostatic LTE models are a stress test.

In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

from Spyctres import plot_xsl_validation_payload

ROOT = Path.cwd().resolve()
if ROOT.name == 'examples':
    ROOT = ROOT.parent
MANIFEST = ROOT / 'examples' / 'xsl_validation_manifest.csv'
RESULTS = ROOT / 'examples' / 'data' / 'xsl_figure1_validation_coarse_results.json'
CACHE = Path('/tmp/spyctres_xsl_cache')
print('Repository:', ROOT)

## Observed-versus-model classification panels

These panels show the continuum-adjusted PHOENIX prediction returned by the actual classification fit. The gray curve is the XSL spectrum, the red curve is the fitted model, and the blue curve is the residual shifted downward by 0.3 for visibility.

The display uses one median scale per target (`plot_scale = "global"`) rather than one independent scale per UVB/VIS/NIR segment. That makes the plot closer to the official XSL full-spectrum display and avoids artificial arm-to-arm jumps caused only by plotting normalization. The segments are still kept separate internally so Spyctres can preserve the documented XSL effective LSF metadata. No extra arm scaling, barycentric correction, or stellar-RV correction is applied by Spyctres for these DR3 products.

In [ ]:
def _global_validation_scale(plot_data):
    chunks = []
    fallback = []
    for segment in plot_data.get('segments', []):
        observed = np.asarray(segment['observed_flux'], dtype=float)
        used = np.asarray(segment.get('used', np.ones(observed.size, dtype=bool)), dtype=bool)
        good = used & np.isfinite(observed)
        if np.any(good):
            chunks.append(observed[good])
        finite = np.isfinite(observed)
        if np.any(finite):
            fallback.append(observed[finite])
    values = np.concatenate(chunks or fallback) if (chunks or fallback) else np.array([1.0])
    scale = np.nanmedian(values)
    return float(scale) if np.isfinite(scale) and scale != 0 else 1.0

if not RESULTS.exists():
    print('No result file yet:', RESULTS)
else:
    payload = json.loads(RESULTS.read_text(encoding='utf-8'))
    rows = payload['results']
    fig, axes = plt.subplots(3, 3, figsize=(16, 11), sharex=False)
    for ax, row in zip(axes.flat, rows):
        plot_data = row.get('validation_plot')
        if not plot_data:
            ax.text(0.5, 0.5, row['status'].replace('_', ' '), ha='center', va='center', transform=ax.transAxes)
            ax.set_axis_off()
            ax.set_title(f"{row['xsl_id']}  {row['spectral_type']}")
            continue
        scale = _global_validation_scale(plot_data)
        for segment_index, segment in enumerate(plot_data['segments']):
            wave = np.asarray(segment['wave_A'], dtype=float)
            observed = np.asarray(segment['observed_flux'], dtype=float)
            model = np.asarray(segment['model_flux'], dtype=float)
            label_observed = 'XSL observed' if segment_index == 0 else None
            label_model = 'PHOENIX fit' if segment_index == 0 else None
            label_residual = 'residual - 0.3' if segment_index == 0 else None
            ax.plot(wave, observed / scale, color='0.45', lw=0.45, label=label_observed)
            ax.plot(wave, model / scale, color='tab:red', lw=0.65, alpha=0.9, label=label_model)
            ax.plot(wave, (observed - model) / scale - 0.3, color='tab:blue', lw=0.4, alpha=0.75, label=label_residual)
        fit = row.get('fit', {})
        ref = row['reference']
        annotation = (
            f"fit: {fit.get('teff', np.nan):.0f} K, {fit.get('logg', np.nan):.2f}, {fit.get('feh', np.nan):.2f}\n"
            f"ref: {ref['teff']:.0f} K, {ref['logg']:.2f}, {ref['feh']:.2f}\n"
            "global plot scale; no Spyctres arm/RV re-correction"
        )
        ax.text(0.02, 0.97, annotation, va='top', transform=ax.transAxes, fontsize=8, bbox={'facecolor': 'white', 'alpha': 0.75, 'edgecolor': 'none'})
        ax.axhline(-0.3, color='0.75', ls=':', lw=0.6)
        ax.set_title(f"{row['xsl_id']}  {row['spectral_type']}  [{row['validation_role']}]", fontsize=10)
        ax.set_ylim(-0.65, 1.45)
        ax.set_xlabel('air wavelength [A]')
        ax.set_ylabel('global-scaled flux')
    axes.flat[1].legend(loc='lower right', fontsize=7)
    fig.suptitle('XSL classification validation: observed spectra and fitted PHOENIX models', y=1.01)
    fig.tight_layout()


## Display-scale comparison for one target

The helper below shows the difference between global target scaling and per-segment diagnostic scaling. `global` is the default for full-spectrum XSL displays. `per_segment` is useful for line-shape inspection, but it must not be interpreted as preserving arm-to-arm continuum levels.


In [ ]:

if not RESULTS.exists():
    print('No result file yet:', RESULTS)
else:
    payload = json.loads(RESULTS.read_text(encoding='utf-8'))
    example = next((row for row in payload['results'] if row.get('validation_plot')), None)
    if example is None:
        print('No validation plot payloads were saved.')
    else:
        for mode in ['global', 'per_segment']:
            fig, axes = plot_xsl_validation_payload(
                example['validation_plot'],
                scale_mode=mode,
                title=f"{example['xsl_id']} display mode: {mode}",
            )
            plt.show()


## Reference sample

The uncertainties are those reported by the source analysis when available. X0013 uses a representative 3400 K value spanning its separate optical/NIR carbon-star solutions and is treated as a diagnostic stress case, not an ordinary PHOENIX recovery target.

In [ ]:
with MANIFEST.open(newline='', encoding='utf-8') as handle:
    stars = list(csv.DictReader(handle))

header = f"{'ID':<6} {'Star':<28} {'Type':<6} {'Teff':>7} {'logg':>7} {'[Fe/H]':>8}  Role"
print(header)
print('-' * len(header))
for star in stars:
    print(f"{star['xsl_id']:<6} {star['star_name'][:28]:<28} {star['spectral_type']:<6} "
          f"{float(star['teff_ref']):7.0f} {float(star['logg_ref']):7.2f} "
          f"{float(star['feh_ref']):8.2f}  {star['validation_role']}")

## Inspect the spectra before fitting

The official DR3 products use air wavelengths in the stellar rest frame. Spyctres preserves each arm's effective Gaussian velocity resolution as segment metadata and does not apply another RV correction. This quick-look plot uses one global display scale per star; if you want the old per-arm line-shape view, use `plot_xsl_validation_payload(..., scale_mode="per_segment")` on a saved validation payload.

In [ ]:
from Spyctres.io import read_spectrum

missing = []
fig, axes = plt.subplots(3, 3, figsize=(15, 10), sharex=True)
for ax, star in zip(axes.flat, stars):
    path = MANIFEST.parent / star['path']
    if not path.exists():
        missing.append(path)
        ax.text(0.5, 0.5, 'missing FITS file', ha='center', va='center')
    else:
        spectrum = read_spectrum(path, instrument='xsl_dr3', warn_unknown=False)
        flux_chunks = []
        for segment in spectrum:
            good = segment.mask & (segment.wave >= 3500) & (segment.wave <= 13000) & np.isfinite(segment.flux)
            if np.any(good):
                flux_chunks.append(segment.flux[good])
        scale = np.nanmedian(np.concatenate(flux_chunks)) if flux_chunks else 1.0
        if not np.isfinite(scale) or scale == 0:
            scale = 1.0
        for segment in spectrum:
            good = segment.mask & (segment.wave >= 3500) & (segment.wave <= 13000)
            if np.any(good):
                ax.plot(segment.wave[good], segment.flux[good] / scale, lw=0.45)
    ax.set_title(f"{star['xsl_id']}  {star['spectral_type']}")
    ax.set_ylim(-0.1, 2.5)
for ax in axes[-1]:
    ax.set_xlabel('air wavelength [A]')
for ax in axes[:, 0]:
    ax.set_ylabel('global-scaled flux')
fig.suptitle('XSL DR3 input spectra: one display scale per star; no Spyctres arm/RV correction', y=1.01)
fig.tight_layout()
if missing:
    print('Missing files:')
    print(*missing, sep='\n')

## Run the classification fits

Set `RUN_FITS=True` when a local PHOENIX library is configured. The default 4000--9000 A interval provides a repeatable optical comparison, avoids the worst DR3 dichroic region, and keeps this example practical. A sparse physical-grid scan selects a local PHOENIX interpolation region before optimization. Manifest roles set the default budget: standard targets receive four local starts, while cool, peculiar, carbon-star, and other diagnostic stress targets receive two bounded starts and do not contribute to ordinary recovery statistics. The runner atomically checkpoints each star and `--resume` skips completed work. This is a broad atmospheric-parameter validation, not yet a precision abundance or rotational-broadening analysis.

In [ ]:
RUN_FITS = False
if RUN_FITS:
    command = [
        sys.executable, str(ROOT / 'scripts' / 'xsl_validation.py'),
        str(MANIFEST), '--output', str(RESULTS), '--resume',
        '--cache-dir', str(CACHE), '--wave-min', '4000', '--wave-max', '9000',
    ]
    print(' '.join(command))
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print('Set RUN_FITS=True to run the classifications.')

## Compare Spyctres with the literature

A reasonable result is not zero residual for every object. Look for recovery across the standard B--K targets, a clean refusal for the O star, and interpretable degradation in the M dwarf, supergiant, and carbon-star stress tests. Large coherent offsets with temperature or gravity are more informative than one isolated outlier.

In [ ]:
if not RESULTS.exists():
    print('No result file yet:', RESULTS)
else:
    payload = json.loads(RESULTS.read_text(encoding='utf-8'))
    fitted = [row for row in payload['results'] if row.get('status') == 'ok']
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    specs = [('teff', 'Teff [K]'), ('logg', 'log g'), ('feh', '[Fe/H]')]
    for ax, (key, label) in zip(axes, specs):
        ref = np.array([row['reference'][key] for row in fitted])
        fit = np.array([row['fit'][key] for row in fitted])
        roles = [row['validation_role'] for row in fitted]
        for role, marker, color in [
            ('standard', 'o', 'tab:blue'),
            ('cool_stress', 's', 'tab:orange'),
            ('peculiar_stress', '^', 'tab:red'),
            ('carbon_star', 'D', 'tab:purple'),
        ]:
            choose = np.array([value == role for value in roles])
            ax.scatter(ref[choose], fit[choose], marker=marker, color=color, label=role)
        lo = min(ref.min(), fit.min())
        hi = max(ref.max(), fit.max())
        ax.plot([lo, hi], [lo, hi], '--', color='0.5', lw=1)
        for x, y, row in zip(ref, fit, fitted):
            ax.annotate(row['xsl_id'], (x, y), fontsize=8, xytext=(3, 3), textcoords='offset points')
        ax.set_xlabel('literature ' + label)
        ax.set_ylabel('Spyctres ' + label)
    axes[0].legend(fontsize=8)
    fig.tight_layout()

    print(f"{'ID':<6} {'status':<20} {'dTeff':>9} {'dlogg':>9} {'d[Fe/H]':>10}")
    for row in payload['results']:
        delta = row.get('delta')
        if delta:
            values = f"{delta['teff']:9.0f} {delta['logg']:9.2f} {delta['feh']:10.2f}"
        else:
            values = ' ' * 30
        print(f"{row['xsl_id']:<6} {row['status']:<20} {values}")

## Interpretation checklist

1. Inspect spectra and fit residuals before interpreting parameter deltas.
2. Evaluate standard B--K targets separately from stress/peculiar targets.
3. Confirm X0116 reports `unsupported_physics`; never extrapolate PHOENIX beyond 12000 K.
4. Treat X0013 disagreement as a test of missing carbon chemistry/C/O, not merely optimizer accuracy.
5. Compare trends above and below 5000 K with the systematics reported by Lançon et al. (2021).
6. Repeat by arm or carefully chosen windows if the joint optical fit shows coherent offsets.
7. Only after these checks decide whether a discrepancy points to masking, LSF handling, continuum treatment, grid coverage, or atmosphere physics.